# 03 — Metadata-first product explorer

**Goal:** read what the product says about itself — structure, units, chunking, acquisition and
processing metadata — **before** touching any pixel. Nothing in this notebook reads image data.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nisar_s1.config import load_kit
from nisar_s1 import product as P

kit = load_kit()                                  # notebooks/kit_config.yaml
OUT = Path(kit["data_dir"]) / "outputs" / "03"   # outputs stay outside the repository
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
session = P.load_session(kit["session_file"], required=("granule", "frequency"))
src_ctx = lambda: P.open_local_or_remote(session["gcov_local"], session["gcov_url"])

In [2]:
with src_ctx() as h5:
    tree = pd.DataFrame(P.hdf5_tree(h5, "/", max_depth=6))
    prof = P.product_profile(h5)
    band = prof.band
    ident = {k: P._txt(v) for k, v in h5[f"science/{band}/identification"].items()}
print(len(tree), "groups and datasets;", (tree.kind == "dataset").sum(), "datasets")
tree["group"] = tree.path.str.rsplit("/", n=1).str[0]
tree[tree.kind == "dataset"].groupby("group").size().rename("datasets").to_frame()

opening local file: NISAR_L2_PR_GCOV_034_144_A_026_2005_DHDH_M_20261103T041937_20261103T042013_P05036_N_F_J_001.h5


123 groups and datasets; 98 datasets


,datasets
group,
/science/LSAR/GCOV/grids/frequencyA,14
/science/LSAR/GCOV/grids/frequencyB,14
/science/LSAR/GCOV/metadata/attitude,4
/science/LSAR/GCOV/metadata/ceosAnalysisReadyData,4
/science/LSAR/GCOV/metadata/orbit,5
/science/LSAR/GCOV/metadata/radarGrid,13
/science/LSAR/GCOV/metadata/sourceData,9
/science/LSAR/identification,35


## Identification: the product's own record of the acquisition

In [3]:
keys = ["granuleId", "missionId", "productType", "productVersion", "radarBand", "orbitPassDirection",
        "lookDirection", "trackNumber", "frameNumber", "zeroDopplerStartTime", "zeroDopplerEndTime",
        "listOfFrequencies", "isGeocoded", "isFullFrame", "processingCenter", "processingType", "productDoi"]
pd.Series({k: ident.get(k, "-") for k in keys}, name="value").to_frame()

,value
granuleId,NISAR_L2_PR_GCOV_034_144_A_026_2005_DHDH_M_202...
missionId,NISAR
productType,GCOV
productVersion,1.0.20
radarBand,L
orbitPassDirection,Ascending
lookDirection,Left
trackNumber,144
frameNumber,26
zeroDopplerStartTime,2026-11-03T04:19:37.000000000


## Layers of the session grid: shape, type, chunking, compression and units

In [4]:
FREQ = session["frequency"]
with src_ctx() as h5:
    gpath = prof.grids[FREQ].path
    info = pd.DataFrame([P.dataset_info(h5[f"{gpath}/{k}"]) for k in h5[gpath]
                         if hasattr(h5[f"{gpath}/{k}"], "shape") and h5[f"{gpath}/{k}"].ndim == 2])
    units = {k: P._txt(h5[f"{gpath}/{k}"].attrs.get("units", b"")) for k in info.path.str.split("/").str[-1]}
info["layer"] = info.path.str.split("/").str[-1]
info["units"] = info.layer.map(units)
info[["layer", "shape", "dtype", "chunks", "compression", "size_GB_uncompressed", "units"]]

opening local file: NISAR_L2_PR_GCOV_034_144_A_026_2005_DHDH_M_20261103T041937_20261103T042013_P05036_N_F_J_001.h5


,layer,shape,dtype,chunks,compression,size_GB_uncompressed,units
0,HHHH,"(17928, 18108)",float32,"(512, 512)",gzip,1.209379,1
1,HVHV,"(17928, 18108)",float32,"(512, 512)",gzip,1.209379,1
2,inputDataExceptionMask,"(17928, 18108)",uint16,"(512, 512)",gzip,0.604690,
3,mask,"(17928, 18108)",uint8,"(512, 512)",gzip,0.302345,
4,numberOfLooks,"(17928, 18108)",float32,"(512, 512)",gzip,1.209379,1
5,rtcGammaToSigmaFactor,"(17928, 18108)",float32,"(512, 512)",gzip,1.209379,1


Each image layer of a 20 m frame is ~1.2 GB uncompressed and is cut in **512 x 512 chunks**:
reading a small window only decompresses the chunks it touches (notebook 05).

## Processing and calibration metadata

In [5]:
with src_ctx() as h5:
    meta = tree[(tree.kind == "dataset") & tree.path.str.contains("metadata|processingInformation")]
    print(len(meta), "metadata datasets, e.g.:")
    for p in meta.path.head(12):
        print("  ", p)
    pi = f"science/{band}/{prof.product_type}/metadata/processingInformation/parameters"
    if pi in h5:
        print("\nprocessing parameters:", list(h5[pi].keys())[:15])

opening local file: NISAR_L2_PR_GCOV_034_144_A_026_2005_DHDH_M_20261103T041937_20261103T042013_P05036_N_F_J_001.h5
35 metadata datasets, e.g.:
   /science/LSAR/GCOV/metadata/attitude/attitudeType
   /science/LSAR/GCOV/metadata/attitude/eulerAngles
   /science/LSAR/GCOV/metadata/attitude/quaternions
   /science/LSAR/GCOV/metadata/attitude/time
   /science/LSAR/GCOV/metadata/ceosAnalysisReadyData/boundingBox
   /science/LSAR/GCOV/metadata/ceosAnalysisReadyData/ceosAnalysisReadyDataDocumentIdentifier
   /science/LSAR/GCOV/metadata/ceosAnalysisReadyData/ceosAnalysisReadyDataProductType
   /science/LSAR/GCOV/metadata/ceosAnalysisReadyData/outputBackscatterDecibelConversionFormula
   /science/LSAR/GCOV/metadata/orbit/interpMethod
   /science/LSAR/GCOV/metadata/orbit/orbitType
   /science/LSAR/GCOV/metadata/orbit/position
   /science/LSAR/GCOV/metadata/orbit/time

processing parameters: ['azimuthIonosphericGeolocationCorrectionApplied', 'dryTroposphericGeolocationCorrectionApplied', 'faradayR

## Optional: bring your own GIS layer (here the study polygons, reprojected to the product CRS)

In [6]:
import geopandas as gpd
sites = gpd.read_file(kit["aoi"]["site_polygons"]).to_crs(prof.grids[FREQ].epsg)
from shapely.geometry import box
inside = sites.within(box(*prof.grids[FREQ].bounds))
sites.assign(inside_grid=inside.values, area_ha=sites.area / 1e4)[["role", "area_ha", "inside_grid"]]

,role,area_ha,inside_grid
0,cut,2.360000,True
1,control,21.819133,True


### Checkpoint
What are the units of `HHHH`? What does `rtcGammaToSigmaFactor` do (hint: notebook 04)? Which DOI
identifies this product?

In [7]:
checks = {
    "metadata read without pixel access": len(tree) > 50,
    "HHHH listed with 512x512 chunks": info.loc[info.layer == "HHHH", "chunks"].iloc[0] == (512, 512),
    "study polygons inside the grid": bool(inside.all()),
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
assert all(checks.values()), "some checks failed: read the messages above"
print("Module 03 STATUS: PASS")

  [x] metadata read without pixel access
  [x] HHHH listed with 512x512 chunks
  [x] study polygons inside the grid
Module 03 STATUS: PASS
